In [1]:
import cv2
from paho.mqtt import client as mqtt_client
import numpy as np
import tools
import random
from yolox.tracker.byte_tracker import BYTETracker
import argparse
import pandas as pd
import torch
import ast
import re


/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Beispielwerte für die Argumente des Trackers
tracker_args = argparse.Namespace(
    track_thresh=0.29,       # Beispielwert für den Tracking-Schwellenwert
    track_buffer=15,        # Beispielwert für den Puffer
    mot20=False,            # Beispielwert für MOT20
    match_thresh=0.9        # Beispielwert für den Matching-Schwellenwert
)

purpose = "testing"
video_path = "input/Test.mp4"
csv = 'input/lang_sam.csv' #"input/output_data.csv"
csv2 = 'input/oneformer.csv'  
output_path = "output/wheeltracker_output.mp4"

tracker = BYTETracker(tracker_args)
tracker2 = BYTETracker(tracker_args)

class Targets:
    def __init__(self):
        self.Auto = []
        self.Einachser = []
        self.Doppelachser = []
        self.Dreifachachser = []
        self.wheels = []

    def check(self, target, bbox, frame_count):
        def bbox_intersection(wheelbox, vehiclebox):
            # wheelbox und vehiclebox: [x1, y1, x2, y2]
            xA = max(wheelbox[0], vehiclebox[0])
            yA = max(wheelbox[1], vehiclebox[1])
            xB = min(wheelbox[2], vehiclebox[2])
            yB = min(wheelbox[3], vehiclebox[3])

            interArea = max(0, xB - xA) * max(0, yB - yA)
            if interArea == 0:
                return 0.0

            wheelboxArea = (wheelbox[2] - wheelbox[0]) * (wheelbox[3] - wheelbox[1])
            #vehicleboxArea = (vehiclebox[2] - vehiclebox[0]) * (vehiclebox[3] - vehiclebox[1])
            intersection = interArea / float(wheelboxArea)
            return intersection
        
        if target.label == "wheel":
            if [target.track_id, frame_count] not in [[t[0], t[1]] for t in self.wheels]:
                self.wheels.append([target.track_id, frame_count, bbox])
                print(f"Target {target.track_id} wurde gezählt.")
            else:
                print(f"Target {target.track_id} ist bereits gezählt.")
                pass

        if target.label == "car":
            if target.track_id not in [t.track_id for t in self.Auto]:

                self.Auto.append(target.track_id)
                self.carcounter = len(self.Auto)
                print(f"Target {target.track_id} wurde gezählt. Aktuelle Anzahl: {self.carcounter}")
            
            else:
                
                print(f"Target {target.track_id} ist bereits gezählt.")
                pass
        
        elif target.label == "truck" or target.label == "bus":
            
            if [target.track_id, frame_count] not in [[t[0], t[1]] for t in self.Einachser]:
                
                self.Einachser.append([target.track_id, frame_count, bbox])
                self.einachsercounter = len(set([t[0] for t in self.Einachser]))
           
            elif target.label == "truck" or target.label == "truck" and target.track_id in [t.track_id for t in self.Einachser]: 
                for einachser in self.Einachser:
                    track_id_e, frame_e, bbox_e = einachser
                    for wheel in self.wheels:
                        track_id_w, frame_w, bbox_w = wheel
                        if frame_w == frame_e:
                            intersection = bbox_intersection(bbox_w, bbox_e)
                            if intersection > 0.5 and [track_id_e, frame_e] not in [[t[0], t[1]] for t in self.Doppelachser]:
                                self.Einachser.remove(einachser)
                                print(f"Target {target.track_id} wurde entfernt, da es sich um einen Doppelachser handelt.")
                                self.Doppelachser.append([[track_id_e, track_id_w], frame_count, [bbox_e, bbox_w]])
                                self.doppelachsercounter = len(set([t[0] for t in self.Doppelachser]))
                                print(f"Target {target.track_id} wurde als Doppelachser gezählt. Aktuelle Anzahl: {self.doppelachsercounter}")
                            else:
                                continue
                        

                self.Doppelachser.append(target.track_id)
                self.doppelachsercounter = len(self.Doppelachser)
                print(f"Target {target.track_id} wurde gezählt. Aktuelle Anzahl: {self.doppelachsercounter}")

            elif target.track_id not in [t.track_id for t in self.Dreifachachser]:
                
                self.Dreifachachser.append(target.track_id)
                self.dreifachachsercounter = len(self.Dreifachachser)
                print(f"Target {target.track_id} wurde gezählt. Aktuelle Anzahl: {self.dreifachachsercounter}")

            self.Doppelachser.append(target)

if purpose == "testing":
    # Read the video file
    cap = cv2.VideoCapture(video_path)
    # Einlesen der CSV-Datei
    data = pd.read_csv(csv, sep=",") #sep ändern, falls nötig
    data2 = pd.read_csv(csv2, sep=",") #für die Interpolation
    

    # Videoeigenschaften abrufen
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec für das Ausgabevideo
    img_size = (width, height)
    frame_max = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video total frames:{frame_max}")
    out = cv2.VideoWriter(output_path, fourcc, fps, img_size)
    
    frame_count = 1 #508

    
    print(f"img_size: {img_size}, Typ: {type(img_size)}")
    print(f"width: {width}, Typ: {type(width)}")
    print(f"height: {height}, Typ: {type(height)}")

    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret or frame is None:
            print("End of video or unable to read the frame.")
            break
        if frame_count >= frame_max:
            print("All data processed.")
            break
        value = data.iloc[frame_count - 1, 1]
        value_count = data.iloc[frame_count - 1, 0]

        print(f"Value at frame {value_count}: {value}")
        if isinstance(value, str):
            skip = value.lower() == 'nan' or value.lower() == 'n/a'
        else:
            skip = pd.isna(value)

        if not skip:
            print(str(data.iloc[frame_count + 1, 1]))
            print("Frame: ", frame_count)
            
            row = data[data['Iteration'] == frame_count].iloc[0]
            if isinstance(row['Label'], (list, np.ndarray)):
                labels = list(row['Label'])
            else:
                labels = ast.literal_eval(row['Label'])

            conf_str = row['Confidence Score']
            if isinstance(conf_str, str):
                # Ersetze alle Whitespaces zwischen Zahlen durch Kommas
                conf_str = re.sub(r'(?<=\d)\s+(?=\d)', ', ', conf_str)
                confidences = ast.literal_eval(conf_str)
            else:
                confidences = ast.literal_eval(row['Confidence Score'])
            confidences = [conf + 0.3 for conf in confidences]
            print(f"Labels: {labels}, Confidences1: {confidences}")
            x_min = ast.literal_eval(row['x_min'])
            y_min = ast.literal_eval(row['y_min'])
            x_max = ast.literal_eval(row['x_max'])
            y_max = ast.literal_eval(row['y_max'])
            for label, conf, xmin, ymin, xmax, ymax in zip(labels, confidences, x_min, y_min, x_max, y_max):
                print(f"Label: {label}, Confidence: {confidences}")
                #bbox = []
                bbox = [xmin, ymin, xmax, ymax]
                print(bbox)
                bbox_and_confidence = torch.tensor([[*bbox, conf]])
                shape = bbox_and_confidence.shape[1]
                print(f"Shape: {shape}")
                img_info = (frame.shape[1], frame.shape[0])
                scale = min(img_size[0] / float(frame.shape[1]), img_size[1] / float(frame.shape[0]))
                print(f"Scale: {scale}")
                online_targets = tracker.update(bbox_and_confidence, img_info, img_size, label)
                # print(f"Tracked Stracks: {len(online_targets.tracked_stracks)}")
                # print(f"Lost Stracks: {len(online_targets.self.lost_stracks)}")
                # print(f"Detections: {len(online_targets.detections)}")
                print(f"Online targets: {online_targets}")
                # online_targets enthält eine Liste von Objekten mit Bounding Boxes und IDs
                for target in online_targets:
                    # Extrahiere die Bounding Box und die ID
                    print(target.track_id, target.label)
                    print(f"Label:{label}")  # ID des Tracks

                    # Konvertieren der Bounding Box in (x1, y1, x2, y2)
                    bbox = target.tlwh
                    x1, y1, w, h = bbox
                    x2, y2 = int(x1 + w), int(y1 + h)
                    x1, y1 = int(x1), int(y1)

                    # Zeichnen der Bounding Box
                    color = (0, 255, 0)  
                    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                    # Zeichnen der Track-ID
                    text = f"ID: {target.label} (OT_{target.track_id})"
                    cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, color, 2)

            del labels, confidences

            row2 = data2[data2['frame'] == frame_count].iloc[0]
            labels2 = row2['labels']
            labels2 = ast.literal_eval(labels2)
            confidences2 = ast.literal_eval(row2['scores'])
            confidences2 = [float(conf) for conf in confidences2]
            bboxes2 = ast.literal_eval(row2['boxes'])
            bboxes2 = np.array(bboxes2, dtype=float)
            blank = True if data2.iloc[frame_count - 1, 1] == '[]' else False
            if not blank:
                for label, conf, bbox in zip(labels2, confidences2, bboxes2):
                        if isinstance(conf, np.ndarray):
                            conf = conf.item()  # Holt den Skalarwert, egal ob 0d oder 1d
                        elif isinstance(conf, list):
                            conf = conf[0]
                        conf = float(conf)
                        print(f"Label: {label}, Confidence: {conf}")
                        if conf > 0.9:
                            bbox_and_confidence = torch.tensor([[*bbox, conf]])
                            img_info = (frame.shape[1], frame.shape[0])
                            scale = min(img_size[0] / float(frame.shape[1]), img_size[1] / float(frame.shape[0]))
                            online_targets2 = tracker2.update(bbox_and_confidence, img_info, img_size, label)
                        
                            for target in online_targets2:
                                print(target.track_id, target.label)
                                bbox = target.tlwh
                                x1, y1, w, h = bbox
                                x2, y2 = int(x1 + w), int(y1 + h)
                                x1, y1 = int(x1), int(y1)

                                color = (0, 255, 0)  
                                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                                text = f"ID: {target.label} (OT_{target.track_id})"
                                cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, color, 2)

                frame_count += 1
                out.write(frame)
            else:
                print("No labels found for this frame.")
                out.write(frame)
                frame_count += 1
        else:
            row = data2[data2['frame'] == frame_count].iloc[0]
            if row['labels'] != '[]':
                labels = ast.literal_eval(row['labels'])
                confidences = ast.literal_eval(row['scores'])
                print(f"Labels: {labels}, Confidences: {confidences}")
                if not isinstance(confidences, list):
                    confidences = [confidences]
                bboxes = ast.literal_eval(row['boxes'])
                for label, conf, bbox in zip(labels, confidences, bboxes):
                    if isinstance(conf, np.ndarray):
                        conf = conf.item()  # Holt den Skalarwert, egal ob 0d oder 1d
                    elif isinstance(conf, list):
                        conf = conf[0]
                    conf = float(conf)
                    print(f"Label: {label}, Confidence: {conf}")
                    if conf > 0.9:
                        bbox_and_confidence = torch.tensor([[*bbox, conf]])
                        img_info = (frame.shape[1], frame.shape[0])
                        scale = min(img_size[0] / float(frame.shape[1]), img_size[1] / float(frame.shape[0]))
                        online_targets2 = tracker2.update(bbox_and_confidence, img_info, img_size, label)
                    
                        for target in online_targets2:
                            print(target.track_id, target.label)
                            bbox = target.tlwh
                            x1, y1, w, h = bbox
                            x2, y2 = int(x1 + w), int(y1 + h)
                            x1, y1 = int(x1), int(y1)

                            color = (0, 255, 0)  
                            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

                            text = f"ID: {target.label} (OT_{target.track_id})"
                            cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 2, color, 2)
                del labels, confidences, bboxes
                out.write(frame)
                frame_count += 1

            else:
                out.write(frame)
                frame_count += 1

    cap.release()
    out.release()

Video total frames:1834
img_size: (1440, 1080), Typ: <class 'tuple'>
width: 1440, Typ: <class 'int'>
height: 1080, Typ: <class 'int'>
Value at frame 1: nan
Labels: ['motorcycle'], Confidences: [0.9352582693099976]
Label: motorcycle, Confidence: 0.9352582693099976
dists []
dists nach mot20 []
matches [] u_track () u_detection (0,)
410 motorcycle
Value at frame 2: nan
Labels: ['motorcycle'], Confidences: [0.9312520027160645]
Label: motorcycle, Confidence: 0.9312520027160645
dists [[0.]]
dists nach mot20 [[0.068748]]
matches [[0 0]] u_track [] u_detection []
410 motorcycle
Value at frame 3: nan
Labels: ['motorcycle'], Confidences: [0.8473859429359436]
Label: motorcycle, Confidence: 0.8473859429359436
Value at frame 4: nan
Labels: ['motorcycle'], Confidences: [0.795390248298645]
Label: motorcycle, Confidence: 0.795390248298645
Value at frame 5: nan
Labels: ['motorcycle'], Confidences: [0.7841203808784485]
Label: motorcycle, Confidence: 0.7841203808784485
Value at frame 6: nan
Labels: ['mot

/workspace/ByteTrack/yolox/tracker/byte_tracker.py:183: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at  ../aten/src/ATen/native/IndexingUtils.h:30.)
  dets_second = bboxes[inds_second]
/workspace/ByteTrack/yolox/tracker/byte_tracker.py:187: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at  ../aten/src/ATen/native/IndexingUtils.h:30.)
  scores_second = scores[inds_second]


Value at frame 25: nan
Value at frame 26: nan
Value at frame 27: nan
Value at frame 28: nan
Value at frame 29: nan
Value at frame 30: nan
Value at frame 31: nan
Value at frame 32: nan
Value at frame 33: nan
Value at frame 34: nan
Value at frame 35: nan
Value at frame 36: nan
Value at frame 37: nan
Value at frame 38: nan
Value at frame 39: nan
Value at frame 40: nan
Value at frame 41: nan
Value at frame 42: nan
Value at frame 43: nan
Value at frame 44: nan
Value at frame 45: nan
Value at frame 46: nan
Value at frame 47: nan
Value at frame 48: nan
Value at frame 49: nan
Value at frame 50: nan
Value at frame 51: nan
Value at frame 52: nan
Value at frame 53: nan
Value at frame 54: nan
Value at frame 55: nan
Value at frame 56: nan
Labels: ['car'], Confidences: [0.8598507642745972]
Label: car, Confidence: 0.8598507642745972
Value at frame 57: nan
Labels: ['car'], Confidences: [0.7909048199653625]
Label: car, Confidence: 0.7909048199653625
Value at frame 58: nan
Labels: ['car'], Confidences: 